# Sistem Peringatan Dini Gejolak Harga Minyak Goreng di Indonesia

Notebook ini mendokumentasikan proses seleksi fitur untuk memprediksi harga mingguan minyak goreng (`Harga_MG`). Tujuan analisis adalah menemukan subset fitur historis yang paling relevan untuk pemodelan prediksi tanpa mengorbankan performa model.

Metodologi mencakup:
- Ekstraksi fitur berdasarkan data historis harga.
- Evaluasi relevansi fitur menggunakan 7 metode seleksi (Filter, Wrapper, dan Embedded).
- Agregasi hasil seleksi dan evaluasi performa model menggunakan rentang waktu kronologis.

## 1. Data Loading dan EDA

### Tujuan
Memuat data historis harga, menyesuaikan format waktu, dan menangani nilai kosong pada observasi mingguan.

### Proses yang Dilakukan
- Melakukan transposisi format dataset mentah menjadi deret waktu mingguan.
- Mengubah tipe data harga ke format numerik.
- Menerapkan interpolasi linear untuk mengisi nilai kosong guna mempertahankan kontinuitas deret waktu.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from sklearn.feature_selection import (f_regression, mutual_info_regression)
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings, os

warnings.filterwarnings('ignore')
# Set working directory ke root workspace
try:
    _nb = globals().get('__vsc_ipynb_file__', '')
    if _nb:
        os.chdir(os.path.dirname(os.path.abspath(_nb)))
    else:
        os.chdir(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass  # Jupyter notebook: cwd sudah benar

DATA_PATH = "dataset/Tabel Harga Minyak Indonesia (2).xlsx"
TARGET = "Harga_MG"

raw = pd.read_excel(DATA_PATH)
harga = raw.drop(columns="No").set_index("Komoditas (Rp)").T
harga.index = pd.to_datetime(harga.index.str.replace(" ", ""), format="%d/%m/%Y")
harga = harga.apply(lambda c: pd.to_numeric(c.astype(str).str.replace(",", "").replace("-", np.nan)))
harga.columns = ["Harga_MG", "Harga_Curah", "Harga_Kemasan1", "Harga_Kemasan2"]

# PERBAIKAN: Interpolasi (bukan dropna) agar time series tidak bolong
harga = harga.interpolate(method='linear')

print(f"Dataset: {harga.shape[0]} minggu, {harga.shape[1]} kolom harga")
display(harga.head())
display(harga.describe().round(0))

## 2. Exploratory Data Analysis

### Tujuan
Menganalisis tren pergerakan harga dan tingkat volatilitas secara visual sebelum tahap rekayasa fitur.

### Proses yang Dilakukan
- Memvisualisasikan harga absolut untuk empat varian minyak goreng (Goreng, Curah, Kemasan 1, Kemasan 2).
- Memvisualisasikan persentase perubahan harga target (`Harga_MG`) pada setiap periode observasi.

### Interpretasi
Grafik menunjukkan seberapa persisten pergerakan harga antarvarian, serta periode-periode yang mengalami lonjakan atau penurunan drastis.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

# Plot harga semua varian
for col in harga.columns:
    axes[0].plot(harga.index, harga[col], label=col, linewidth=1.2)
axes[0].set_title("Harga Mingguan Minyak Goreng (PIHPS)", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Harga (Rp)")
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Plot perubahan persen target
pct = harga[TARGET].pct_change() * 100
axes[1].bar(harga.index, pct, color=np.where(pct >= 0, 'steelblue', 'salmon'), width=5)
axes[1].set_ylabel("Perubahan (%)")
axes[1].set_xlabel("Tanggal")
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Rentang data: {harga.index.min().date()} s/d {harga.index.max().date()}")
print(f"Missing values setelah interpolasi: {harga.isnull().sum().sum()}")

### Analisis Grafik Harga (lanjutan)
---
- Terlihat bahwa pergerakan harga keempat varian minyak goreng (Goreng, Curah, Kemasan 1, Kemasan 2) sangat mirip dan searah dari waktu ke waktu.
- Grafik lonjakan (bawah) menunjukkan ada periode tertentu di mana harga sangat fluktuatif (bar panjang), namun sebagian besar waktu harganya cenderung stabil.

## 3. Feature Engineering

### Tujuan
Mentransformasi observasi harga historis menjadi fitur prediktif yang dapat digunakan oleh model supervised learning tanpa menyebabkan kebocoran informasi (data leakage).

### Konsep
Fitur turunan seperti nilai periode sebelumnya (`Lag`) dan rata-rata bergerak (`Roll_Mean`) berguna dalam peramalan deret waktu. Penggunaan harga pada waktu `t` dari varian lain untuk memprediksi harga varian utama pada waktu `t` dapat menyebabkan data leakage jika direkam dalam periode bersamaan.

### Proses yang Dilakukan
- Mengekstraksi fitur `Lag_1` hingga `Lag_4` dari target utama.
- Mengekstraksi fitur `Lag_1` dari varian harga lainnya.
- Menghitung nilai rata-rata dan simpangan baku bergerak (`Roll_Mean_4`, `Roll_Std_4`), serta perubahan persentase (`Pct_Change_1`).
- Menambahkan fitur komponen waktu dan indikator biner hari besar keagamaan.

### Catatan Penting
Seluruh observasi dengan nilai kosong (NaN) yang diakibatkan oleh proses penggeseran nilai historis dihapus secara selektif.

In [ ]:
s = harga[TARGET]
fitur = pd.DataFrame(index=harga.index)

# Lag features historis target
fitur["Lag_1"] = s.shift(1)
fitur["Lag_2"] = s.shift(2)
fitur["Lag_3"] = s.shift(3)   # BARU: ditambahkan Lag_3
fitur["Lag_4"] = s.shift(4)

# PERBAIKAN: Lag features untuk varian lain (hindari data leakage)
fitur["Curah_Lag_1"]    = harga["Harga_Curah"].shift(1)
fitur["Kemasan1_Lag_1"] = harga["Harga_Kemasan1"].shift(1)
fitur["Kemasan2_Lag_1"] = harga["Harga_Kemasan2"].shift(1)

# Rolling features & Perubahan Pct
fitur["Roll_Mean_4"]  = s.shift(1).rolling(4).mean()
fitur["Roll_Std_4"]   = s.shift(1).rolling(4).std()
fitur["Pct_Change_1"] = s.shift(1).pct_change()

# Fitur Waktu
fitur["Bulan"]     = fitur.index.month
fitur["Minggu_ke"] = fitur.index.isocalendar().week.astype(int)
fitur["Tahun"]     = fitur.index.year

# Fitur Libur Keagamaan
fitri = ["2022-05-02", "2023-04-22", "2024-04-10", "2025-03-31", "2026-03-21"]
libur = {
    "Ramadan":    (fitri, -30, -1),
    "Idul_Fitri": (fitri, -7, 7),
    "Idul_Adha":  (["2022-07-09", "2023-06-29", "2024-06-17", "2025-06-06", "2026-05-27"], -7, 3),
    "Nyepi":      (["2022-03-03", "2023-03-22", "2024-03-11", "2025-03-29", "2026-03-19"], -3, 1),
    "Natal":      (["2022-12-25", "2023-12-25", "2024-12-25", "2025-12-25", "2026-12-25"], -7, 3),
}

pekan_akhir = fitur.index + pd.Timedelta(days=6)
for nama, (tanggal, bawah, atas) in libur.items():
    fitur[nama] = 0
    for d in pd.to_datetime(tanggal):
        fitur[nama] |= ((fitur.index <= d + pd.Timedelta(days=atas)) &
                        (pekan_akhir >= d + pd.Timedelta(days=bawah))).astype(int)

# Drop NaN akibat efek Lagging
df = fitur.assign(Harga_MG=s).dropna()
X = df.drop(columns="Harga_MG")
y = df["Harga_MG"]

display(Markdown(f"**Regresi Time Series** — {X.shape[0]} sampel, {X.shape[1]} fitur."))
display(Markdown("**Daftar fitur:**"))
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

## 4. Univariate Feature Selection

### Tujuan
Menilai seberapa kuat keterikatan langsung antara masing-masing fitur tunggal dengan target.

### Konsep
- **F-score**: Mengukur signifikansi hubungan linear antara fitur dan target dalam konteks regresi.
- **Mutual Information**: Mengukur ketidakpastian informasi target yang tereduksi akibat suatu fitur, dan sanggup menangkap hubungan nonlinier.

### Interpretasi
Skor yang lebih tinggi menunjukkan indikasi relevansi langsung menurut metode tersebut, namun bukan bukti kausalitas ataupun jaminan performa prediksi multivariat.

In [ ]:
uni = pd.DataFrame({
    "F_score": f_regression(X, y)[0],
    "Mutual_Info": mutual_info_regression(X, y, random_state=42)
}, index=X.columns).sort_values("F_score", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

uni["F_score"].sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title("Skor F (Univariate)", fontweight='bold')
axes[0].set_xlabel("F-score")

uni["Mutual_Info"].sort_values().plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title("Mutual Information", fontweight='bold')
axes[1].set_xlabel("MI Score")

plt.tight_layout()
plt.show()

display(Markdown("### Tabel Skor Univariate"))
display(uni.round(3))

### Hasil Seleksi Univariate (lanjutan)
---
- Dari grafik skor F dan Mutual Information, terlihat bahwa fitur harga minggu lalu (`Lag_1`) memiliki bar paling panjang, menandakan hubungan terkuat dengan harga minggu ini.
- Fitur hari libur keagamaan (seperti Idul Fitri, Natal, dll) memiliki bar terpendek, berarti secara individu fitur-fitur tersebut kurang kuat untuk dijadikan patokan tebakan level harga.

## 5. Random Forest Feature Importance

### Tujuan
Mengevaluasi kepentingan fitur berdasarkan kontribusinya pada penurunan tingkat kesalahan ansambel pohon keputusan.

### Konsep
Random Forest menghitung importance dengan mengukur seberapa besar sebuah fitur mampu menurunkan impurity (error prediksi) setiap kali ia digunakan sebagai pemisah (split) simpul (node).

### Interpretasi
Nilai yang lebih besar menandakan fitur sering diandalkan oleh model dalam mendeskriminasi data.

### Catatan Penting
Metode ini dapat memberikan bobot berlebih (bias) pada fitur yang memiliki kardinalitas tinggi atau rentang distribusi nilai yang lebar.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X, y)

imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(7, max(4, len(imp) * 0.35)))
imp.plot.barh(ax=ax, color='forestgreen')
ax.set_title("Feature Importance (Random Forest)", fontweight='bold')
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

display(imp.sort_values(ascending=False).round(4).to_frame("RF_Importance"))

### Hasil Random Forest Importance (lanjutan)
---
- Sama seperti pengujian sebelumnya, model algoritma Random Forest juga paling sering mengandalkan fitur `Lag_1` untuk membuat tebakan yang akurat.
- Fitur rata-rata pergerakan harga (`Roll_Mean`) juga dianggap lumayan penting, tetapi skornya masih jauh di bawah kehebatan `Lag_1`.

## 6. Permutation Importance

### Tujuan
Mengukur signifikansi prediktif fitur secara empiris dengan mengacak informasi aslinya.

### Konsep
Nilai pada sebuah fitur diacak secara artifisial, dan penurunan performa model dicatat. Jika model memiliki dependensi absolut pada fitur tersebut, pengacakan akan menyebabkan pelemahan metrik evaluasi yang sangat masif.

### Interpretasi
Rata-rata penurunan akurasi (Importance Mean) yang besar menandakan fitur krusial. Nilai di sekitar nol berarti fitur tidak digunakan oleh model, dan penghapusannya tidak akan mengganggu performa model awal.

In [ ]:
tscv = TimeSeriesSplit(5)
perm = permutation_importance(rf_model, X, y, n_repeats=30,
                               random_state=42, scoring='r2', n_jobs=-1)

perm_imp = pd.DataFrame({
    'Importance_Mean': perm.importances_mean,
    'Importance_Std': perm.importances_std
}, index=X.columns).sort_values('Importance_Mean')

fig, ax = plt.subplots(figsize=(7, max(4, len(perm_imp) * 0.35)))
ax.barh(perm_imp.index, perm_imp['Importance_Mean'],
        xerr=perm_imp['Importance_Std'], color='darkorchid', capsize=3)
ax.set_title("Permutation Importance", fontweight='bold')
ax.set_xlabel("Mean R² Decrease")
ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()

display(Markdown("Fitur dengan importance negatif/nol bisa dihapus tanpa rugi."))
display(perm_imp.sort_values('Importance_Mean', ascending=False).round(4))

### Hasil Permutation Importance (lanjutan)
---
- Saat data fitur `Lag_1` sengaja diacak oleh sistem, error model langsung melonjak sangat tajam (bar paling kanan), membuktikan bahwa fitur ini adalah tulang punggung tebakan model.
- Beberapa fitur di bagian bawah memiliki nilai di sekitar nol, yang berarti ada atau tidaknya fitur tersebut tidak mengubah performa tebakan model sama sekali (bisa dibuang).

## 7. Correlation Matrix & Heatmap

### Tujuan
Mendeteksi redundansi informasi linear antarfitur dan mengobservasi arah hubungan fitur terhadap target.

### Konsep
Korelasi linear (Pearson) beroperasi pada skala -1 hingga 1. Korelasi antarfitur yang ekuivalen (mendekati skala maksimal) menandakan adanya duplikasi eksplanatorik yang bisa diminimalkan.

### Interpretasi
Korelasi tinggi mengindikasikan redundansi komputasional, meski tidak selalu mendegradasi performa model berbasis himpunan pohon secara ekstrem.

In [ ]:
corr = X.assign(target=y).corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.zeros_like(corr, dtype=bool)
sns.heatmap(corr, cmap="coolwarm", center=0,
            annot=len(X.columns) <= 20, fmt=".2f",
            linewidths=0.5, ax=ax, square=True,
            cbar_kws={"shrink": 0.8})
ax.set_title("Matriks Korelasi (Fitur + Target)", fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Deteksi pasangan redundan
fc = X.corr().abs()
upper = fc.where(np.triu(np.ones(fc.shape, bool), k=1))
pasangan = upper.stack().loc[lambda v: v > 0.8].sort_values(ascending=False)

display(Markdown(f"### Pasangan fitur redundan (korelasi > 0.8): **{len(pasangan)} pasangan**"))
display(pasangan.rename("Korelasi").round(3).to_frame())

### Hasil Matriks Korelasi (lanjutan)
---
- Perhatikan pada heatmap, kotak-kotak berwarna merah tua menunjukkan hubungan antar-fitur yang sangat identik atau kembar (korelasi > 0.8).
- Terlihat jelas bahwa fitur turunan harga (seperti varian kemasan dan curah) saling berkorelasi sangat kuat satu sama lain. Kita hanya perlu mengambil salah satu saja agar datanya tidak mubazir.

## 8. Variance Inflation Factor (VIF)

### Tujuan
Mengevaluasi derajat multikolinearitas eksak melalui kuantifikasi gabungan seluruh variabel bebas independen.

### Konsep
VIF menghitung sejauh mana varians koefisien estimasi membesar diakibatkan interkorelasi suatu variabel prediktor dengan variabel prediktor lain secara menyeluruh, tidak hanya sebatas hubungan univariat.

### Interpretasi
- **VIF > 5-10**: Menandakan redundansi signifikan (multikolinearitas struktural parah).
- **Catatan**: VIF tinggi bukan parameter utilitas presisi fitur; hal ini hanya memproyeksikan duplikasi.

In [ ]:
# Hitung VIF untuk semua fitur
vif_data = pd.DataFrame({
    "Fitur": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
}).sort_values("VIF", ascending=False).reset_index(drop=True)

# Kategori VIF
def vif_label(v):
    if v > 10: return "TINGGI (> 10)"
    elif v > 5: return "SEDANG (5-10)"
    elif v > 1: return "RENDAH (1-5)"
    else: return "MINIMAL (≤ 1)"

vif_data["Kategori"] = vif_data["VIF"].apply(vif_label)

# Visualisasi
fig, ax = plt.subplots(figsize=(8, max(4, len(vif_data) * 0.35)))
colors = vif_data["VIF"].apply(lambda v: 'crimson' if v > 10 else ('orange' if v > 5 else 'seagreen'))
ax.barh(vif_data["Fitur"], vif_data["VIF"], color=colors)
ax.axvline(5, color='orange', linewidth=1.5, linestyle='--', label='Batas Sedang (5)')
ax.axvline(10, color='red', linewidth=1.5, linestyle='--', label='Batas Tinggi (10)')
ax.set_title("Variance Inflation Factor (VIF)", fontweight='bold')
ax.set_xlabel("VIF")
ax.legend()
plt.tight_layout()
plt.show()

display(vif_data.round(2))
print(f"\nFitur dengan VIF > 10 (sangat redundan): {vif_data.loc[vif_data['VIF'] > 10, 'Fitur'].tolist()}")

### Hasil Variance Inflation Factor (lanjutan)
---
- Fitur yang melewati garis putus-putus merah (VIF > 10) adalah fitur yang informasinya bertumpang tindih parah dengan fitur lain (menyontek).
- Banyak fitur turunan (seperti `Roll_Mean` dan `Lag_2` ke atas) terkena penalti VIF tinggi karena inti informasinya sebenarnya sudah terwakili cukup oleh `Lag_1`.

## 9. Recursive Feature Elimination (RFECV)

### Tujuan
Menyusutkan ruang dimensi variabel menggunakan skema eliminasi regresif berulang.

### Konsep
Evaluasi diawali dengan himpunan penuh seluruh fitur, dan fitur inferior dibuang secara bertahap. Estimasi evaluasi (R²) untuk menemukan jumlah optimal disahkan melalui pembagian lipatan kronologis (Cross-Validation).

### Interpretasi
Jumlah fitur terbaik ditentukan berdasarkan puncak kurva evaluasi CV, dan urutan hierarki eliminasi direpresentasikan dalam bentuk urutan ranking (1 merepresentasikan kualitas terbaik).

In [ ]:
selector = RFECV(
    estimator=RandomForestRegressor(n_estimators=100, random_state=42),
    step=1,
    cv=TimeSeriesSplit(5),
    scoring='r2',
    n_jobs=-1,
    min_features_to_select=3
)
selector.fit(X, y)

rfe_ranking = pd.Series(selector.ranking_, index=X.columns)
rfe_support = pd.Series(selector.support_, index=X.columns)

# Plot jumlah fitur vs skor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Skor vs jumlah fitur
n_features_range = range(selector.min_features_to_select, len(X.columns) + 1)
axes[0].plot(n_features_range, selector.cv_results_['mean_test_score'], 'b-o', markersize=4)
axes[0].fill_between(n_features_range,
                     selector.cv_results_['mean_test_score'] - selector.cv_results_['std_test_score'],
                     selector.cv_results_['mean_test_score'] + selector.cv_results_['std_test_score'],
                     alpha=0.2)
axes[0].axvline(selector.n_features_, color='red', linestyle='--',
                label=f'Optimal: {selector.n_features_} fitur')
axes[0].set_xlabel("Jumlah Fitur")
axes[0].set_ylabel("R² (Cross-Validation)")
axes[0].set_title("RFECV: Skor vs Jumlah Fitur", fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Ranking fitur
rfe_ranking.sort_values().plot.barh(ax=axes[1], color='teal')
axes[1].set_title("RFE Feature Ranking (1 = terbaik)", fontweight='bold')
axes[1].set_xlabel("Ranking")

plt.tight_layout()
plt.show()

rfe_selected = rfe_support[rfe_support].index.tolist()
print(f"Jumlah fitur optimal (RFECV): {selector.n_features_}")
print(f"Fitur terpilih: {rfe_selected}")

### Hasil Recursive Feature Elimination (lanjutan)
---
- **Grafik Garis (kiri):** Menunjukkan bahwa kita tidak perlu menggunakan seluruh belasan fitur; cukup menggunakan sekitar separuhnya saja, akurasi model sudah bisa mencapai titik puncaknya.
- **Grafik Batang (kanan):** Mengurutkan pemenang akhirnya. Fitur dengan ranking 1 (bar terpendek) adalah daftar fitur elit yang wajib kita pertahankan.

## 10. LASSO Feature Selection

### Tujuan
Menemukan variabel yang tidak signifikan dan memangkas distribusinya menggunakan penalti mutlak dalam fungsi kerugian regresi (L1 Regularization).

### Konsep
Penalti absolut pada optimisasi fungsi loss memaksa koefisien fitur yang minim kontribusi menjadi bernilai nol presisi (penyusutan variabel penuh).

### Catatan Penting
Konvergensi koefisien bergantung secara absolut terhadap standarisasi dimensi nilai (skala seragam), sehingga prapemrosesan (`StandardScaler`) diberlakukan secara imperatif.

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

lasso = LassoCV(cv=TimeSeriesSplit(5), random_state=42, max_iter=10000)
lasso.fit(X_scaled, y)

lasso_coef = pd.Series(np.abs(lasso.coef_), index=X.columns).sort_values(ascending=False)
lasso_selected = lasso_coef[lasso_coef > 0].index.tolist()
lasso_eliminated = lasso_coef[lasso_coef == 0].index.tolist()

# Visualisasi
fig, ax = plt.subplots(figsize=(8, max(4, len(lasso_coef) * 0.35)))
colors = ['steelblue' if c > 0 else 'lightgray' for c in lasso_coef.sort_values()]
lasso_coef.sort_values().plot.barh(ax=ax, color=colors)
ax.set_title(f"LASSO Coefficients (alpha={lasso.alpha_:.4f})", fontweight='bold')
ax.set_xlabel("|Koefisien| (standardized)")
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

display(Markdown(f"**Fitur terpilih LASSO ({len(lasso_selected)}):** {lasso_selected}"))
display(Markdown(f"**Fitur dieliminasi ({len(lasso_eliminated)}):** {lasso_eliminated}"))
display(lasso_coef.round(4).to_frame("LASSO_Coef"))

### Hasil Seleksi LASSO (lanjutan)
---
- Bar berwarna biru menunjukkan deretan fitur yang masih diberi nilai dan dipertahankan oleh model LASSO.
- Bar berwarna abu-abu yang persis berada di garis angka 0 menunjukkan fitur yang dinilai benar-benar tidak berguna dan koefisiennya dipangkas habis (dibuang) oleh sistem LASSO.

## 11. Feature Ranking Aggregation

### Tujuan
Membangun kesepakatan akhir rasional dengan menggabungkan peringkat dari seluruh mekanisme evaluasi sebelumnya.

### Proses yang Dilakukan
- Peringkat akhir diagregasi menjadi mean (rata-rata unweighted) untuk menetralisir bias algoritma tunggal.
- Seluruh observasi fitur diurutkan (ascending).
- Fitur inferior berperingkat lebih rendah dieliminasi secara sistematis jika secara spesifik redundan (Korelasi > 0.8) terhadap fitur peringkat dominan yang tersisa.

In [ ]:
# Gabungkan skor dari semua metode
rank = pd.DataFrame({
    "F_score":    uni["F_score"],
    "MI":         uni["Mutual_Info"],
    "RF_Imp":     imp,
    "Perm_Imp":   perm_imp["Importance_Mean"],
    "RFE_Rank":   rfe_ranking,
    "LASSO_Coef": lasso_coef,
}).rank(ascending=False)

# Untuk RFE_Rank, ranking=1 sudah terbaik, jadi invert
rank["RFE_Rank"] = rfe_ranking.rank(ascending=True)

rank["Avg_Rank"] = rank.mean(axis=1)
rank = rank.sort_values("Avg_Rank")

# Filter redundansi
tersisa = []
for c in rank.index:
    if all(fc.loc[c, t] <= 0.8 for t in tersisa):
        tersisa.append(c)
redundant = [c for c in rank.index if c not in tersisa]

K = max(3, len(X.columns) // 2)
selected = tersisa[:K]

display(Markdown("### Ranking Gabungan Semua Metode"))
display(rank.round(2))

display(Markdown(f"### Fitur redundan yang dibuang ({len(redundant)}):"))
for f in redundant:
    pasangan_terkait = [t for t in tersisa if fc.loc[f, t] > 0.8]
    print(f"  ✗ {f} — redundan dengan {pasangan_terkait}")

display(Markdown(f"### ✅ Fitur Terpilih ({len(selected)}):"))
for i, f in enumerate(selected, 1):
    print(f"  {i}. {f} (avg rank: {rank.loc[f, 'Avg_Rank']:.1f})")

## 12. Visualisasi Rangkuman Metode

### Tujuan
Menyediakan tinjauan visual holistik dan hierarkis dari hasil rekapitulasi akhir klasemen variabel.

### Interpretasi
Warna dan skala matriks membedakan distribusi konsensus tiap metode. Skema bar pada klasemen akhir mengisolasi fitur terpilih secara demarkatif (hijau) dari subset variabel yang di-filter.

In [ ]:
# Heatmap ranking semua metode
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Heatmap ranking
rank_display = rank.drop(columns="Avg_Rank").loc[rank.index]
sns.heatmap(rank_display, annot=True, fmt=".0f", cmap="YlOrRd_r",
            linewidths=0.5, ax=axes[0], cbar_kws={"label": "Ranking (1=terbaik)"})
axes[0].set_title("Ranking Fitur per Metode", fontweight='bold')
axes[0].set_xlabel("Metode Seleksi")

# Plot 2: Bar chart rata-rata ranking
colors = ['forestgreen' if f in selected else ('orange' if f in redundant else 'lightgray')
          for f in rank.sort_values("Avg_Rank").index]
rank.sort_values("Avg_Rank")["Avg_Rank"].plot.barh(ax=axes[1], color=colors)
axes[1].set_title("Rata-rata Ranking (hijau = terpilih, oranye = redundan)", fontweight='bold')
axes[1].set_xlabel("Avg Rank (rendah = lebih baik)")
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()

### Kesimpulan Seleksi Fitur (lanjutan)
---
- **Grafik Bar (kanan)** merangkum pemenang akhir dari semua pengujian: Batang berwarna **Hijau** adalah fitur terbaik yang lolos seleksi dan tidak duplikat.
- Fitur yang berwarna Oranye atau Abu-abu adalah fitur yang dinyatakan gugur karena skornya rendah dari awal, atau sekadar mengulang informasi dari fitur hijau.

## 13. Feature Selection Evaluation

### Tujuan
Memvalidasi klaim efisiensi subset dengan membandingkan kualitas empiris ruang himpunan seluruh dimensi (sebelum) vs himpunan seleksi akhir (sesudah).

### Konsep
- Modul `TimeSeriesSplit` mengakomodasi asimetri pengujian rentang waktu observasional secara natural dengan mempertahankan urutan kronologis historis (tanpa pengacakan masa depan vs masa lalu).
- Akurasi (R²) mengukur kecocokan peramalan varians target.

### Interpretasi
Nilai R² himpunan terseleksi yang ekuivalen maupun memiliki delta peningkatan marjinal menunjukkan kesuksesan proses redundansi dimensi variabel terhadap model peramalan.

In [ ]:
est = RandomForestRegressor(random_state=42)
cv = TimeSeriesSplit(5)

results = {}
for name, cols in {"Semua fitur": X.columns.tolist(), "Terseleksi": selected}.items():
    scores = cross_val_score(est, X[cols], y, cv=cv, scoring="r2")
    results[name] = {"R² Mean": scores.mean(), "R² Std": scores.std(),
                     "Jumlah Fitur": len(cols)}

result_df = pd.DataFrame(results).T
display(Markdown("### Hasil Validasi (TimeSeriesSplit, 5-fold)"))
display(result_df.round(4))

diff = results["Terseleksi"]["R² Mean"] - results["Semua fitur"]["R² Mean"]
fewer = len(X.columns) - len(selected)
print(f"\nSeleksi fitur mengurangi {fewer} fitur ({len(X.columns)} → {len(selected)})")
print(f"Perubahan R²: {diff:+.4f}")
if diff >= -0.01:
    print("✅ Performa terjaga/meningkat dengan fitur yang lebih sedikit!")
else:
    print("⚠️ Ada penurunan performa, pertimbangkan untuk menambah fitur terpilih.")

## 14. Export Dataset Terseleksi

### Tujuan
Menyimpan matriks dataset persisten yang telah terfiltrasi secara definitif.

### Proses yang Dilakukan
Dataset dikonkatenasi dengan nilai observasi target `Harga_MG` pada indeks yang koheren, dan disalurkan secara statis menuju folder rujukan kompilasi.

In [ ]:
output_path = "output/dataset_terseleksi.csv"
os.makedirs("output", exist_ok=True)
X[selected].assign(**{TARGET: y}).to_csv(output_path, index=True)
print(f"Dataset terseleksi disimpan ke: {output_path}")
print(f"Shape: {X[selected].shape[0]} baris, {len(selected)} fitur + 1 target")

## 15. Analisis Hasil Seleksi Fitur

### Rangkuman Observasi Akhir

1. **Efek Ketergantungan Historis (Lag)**
   Distribusi matriks `Lag_1` secara imperatif mendominasi spektrum signifikansi setiap metode evaluasi analitik, membuktikan sifat autokorelasi (momentum historis deterministik) dari komoditas target utama.
2. **Kolinearitas Antarvarian Harga**
   Fluktuasi nilai dari varian harga substitutif (Curah dan Kemasan) secara empiris tersinkronisasi ekstrem terhadap varian target. Metode korelatif murni menyingkirkan elemen duplikatif ini untuk mendemokrasikan ruang model dari redundansi laten.
3. **Redundansi Ekspresi Deviasi**
   Fitur matematis perataan (`Roll_Mean`) serta urutan masa tertinggal yang lebih makro (`Lag_2`+) terbukti menempati matriks penalti VIF kronis, karena observasi tersebut mengulang porsi informasi yang telah tersubtitusi secara penuh pada indeks historis paling proksimal.
4. **Pembuktian Validitas Empiris**
   Studi eksperimental dengan skema validasi rentang waktu iteratif (CV) membuktikan stabilitas dan kelayakan operasional dari metodologi agregat seleksi yang diterapkan.